In [1]:
import argparse
#import logging
import sys
import pandas as pd
from pathlib import Path
from loguru import logger
import json

In [16]:
project_root = Path.cwd().parents[2]#.parent.parent
print(project_root)
input = project_root / "data" / "raw" / "Telco-Customer-Churn.csv"
output = project_root / "data" / "data_quality" / "processed_data" / "Telco_Customer_Churn_interim.csv"


c:\Users\steph\Documents\GitHub\ML_pipe


In [3]:
logger.remove() #remove default loguru handler

# function to return a json object with select fields from the log record
def serialize(record):
    subset = {
        "level": record["level"].name,
        "timestamp": record["time"].timestamp(),
        "message": record["message"]}
    return json.dumps(subset)

In [4]:
# function for setting formatting of log messages:
def formatter(record):
    record["extra"]["serialized"] = serialize(record)
    return "{extra[serialized]}\n"

logger.add(sys.stderr, format=formatter, level="INFO")

1

extract data from .csv

In [17]:
def extract_data(file_path: Path) -> pd.DataFrame:
    """Reads raw data from a CSV file."""
    logger.info(f"Starting extraction from: {file_path}")
    
    if not file_path.exists():
        raise FileNotFoundError(f"Input file not found at {file_path}")
        
    df = pd.read_csv(file_path)
    logger.info(f"Successfully extracted {len(df)} rows and {len(df.columns)} columns.")
    return df

In [18]:
raw_df = extract_data(input)

{"level": "INFO", "timestamp": 1790534447.25595, "message": "Starting extraction from: c:\\Users\\steph\\Documents\\GitHub\\ML_pipe\\data\\raw\\Telco-Customer-Churn.csv"}
{"level": "INFO", "timestamp": 1790534447.386319, "message": "Successfully extracted 7043 rows and 21 columns."}


In [19]:
####
# 3. Transform data
####

def transform_data(df: pd.DataFrame) -> pd.DataFrame:
    """Cleans and transforms the raw dataframe."""
    logger.info("Starting data transformation...")
    
    # Make a copy 
    df_clean = df.copy()
    
    # Fix the data issue blank spaces in TotalCharges
    numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
    for col in numeric_cols:
        if col in df_clean.columns:
            df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
            

    # Drop exact duplicate rows if any exist
    initial_count = len(df_clean)
    df_clean = df_clean.drop_duplicates()
    logger.info(f"Dropped {initial_count - len(df_clean)} duplicate rows.")
    
    logger.info("Transformation complete.")
    return df_clean


In [22]:
clean_data = transform_data(raw_df)
clean_data.head(5)
clean_data.info()

{"level": "INFO", "timestamp": 1790534826.865936, "message": "Starting data transformation..."}
{"level": "INFO", "timestamp": 1790534826.914902, "message": "Dropped 0 duplicate rows."}
{"level": "INFO", "timestamp": 1790534826.917164, "message": "Transformation complete."}


<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [24]:
####
# 4. Load data
####


def upload_data(df: pd.DataFrame, output_path: Path) -> None:
    """Saves the transformed dataframe to the destination."""
    logger.info(f"Starting upload to: {output_path}")

    output_path.parent.mkdir(parents = True, exist_ok = True)
    
    if output_path.suffix == ".parquet":
        df.to_parquet(output_path, index=False)
    else:
        df.to_csv(output_path, index=False)
 
    logger.info(f"Successfully uploaded {len(df)} rows and {len(df.columns)} columns to {output_path}.")

In [26]:
####
# 5.Main execution and CLI
####

def parse_arguments() -> argparse.Namespace:
    """Sets up command-line arguments."""
    parser = argparse.ArgumentParser(description="Run ETL Pipeline")
    parser.add_argument(
        "--input", 
        type=Path, 
        required=True, 
        help="Path to the raw input CSV file"
    )
    parser.add_argument(
        "--output", 
        type=Path, 
        required=True, 
        help="Path to save the cleaned output file (CSV or Parquet)"
    )
    return parser.parse_args()

In [ ]:
def main() -> None:
    """Main orchestrator for the ETL pipeline."""
    args = parse_arguments()
    
    try:
        # Orchestrate the pipeline
        # raw_df = extract_data(args.input)
        # clean_df = transform_data(raw_df)
        # upload_data(clean_df, args.output)
        raw_df = extract_data(input)
        clean_df = transform_data(raw_df)
        upload_data(clean_df, output)
        
        logger.info("ETL Pipeline completed successfully.")
        
    except Exception as e:
        logger.error(f"ETL Pipeline failed with error: {e}", exc_info=True)
        # Exit with a non-zero status code so orchestrators (like Airflow) know it failed
        sys.exit(1) 

{"level": "INFO", "timestamp": 1790536779.798859, "message": "Starting upload to: c:\\Users\\steph\\Documents\\GitHub\\ML_pipe\\data\\data_quality\\processed_data\\Telco_Customer_Churn_interim.csv"}
{"level": "INFO", "timestamp": 1790536779.941114, "message": "Successfully uploaded 7043 rows and 21 columns to c:\\Users\\steph\\Documents\\GitHub\\ML_pipe\\data\\data_quality\\processed_data\\Telco_Customer_Churn_interim.csv."}
